<a href="https://colab.research.google.com/github/Romeseven-HT/Artificial_Intelligence/blob/main/b%C3%A0i_t%E1%BA%ADp_v%E1%BB%81_nh%C3%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================
# BÀI 1: GIẢI BÀI TOÁN 15-PUZZLE BẰNG THUẬT TOÁN A*
# ================================================

import heapq
from copy import deepcopy

class Node:
    def __init__(self, board, parent=None, move="", depth=0, cost=0):
        self.board = board          # Trạng thái bàn cờ (list 2D)
        self.parent = parent        # Node cha
        self.move = move            # Nước đi dẫn đến node này
        self.depth = depth          # Độ sâu (số nước đi)
        self.cost = cost            # g(n) = depth
        self.heuristic = 0          # h(n)
        self.f = 0                  # f(n) = g(n) + h(n)

    def __lt__(self, other):
        return self.f < other.f

def find_blank(board):
    """Tìm vị trí ô trống (0)"""
    for i in range(len(board)):
        for j in range(len(board[0])):
            if board[i][j] == 0:
                return i, j
    return None

def manhattan_distance(board):
    """Heuristic: Manhattan Distance (rất tốt cho 15-Puzzle)"""
    n = len(board)
    distance = 0
    for i in range(n):
        for j in range(n):
            tile = board[i][j]
            if tile != 0:
                # Vị trí đích của tile
                target_x = (tile - 1) // n
                target_y = (tile - 1) % n
                distance += abs(i - target_x) + abs(j - target_y)
    return distance

def is_goal(board):
    """Kiểm tra bàn cờ đã về trạng thái đích chưa"""
    n = len(board)
    count = 1
    for i in range(n):
        for j in range(n):
            if board[i][j] != count and not (i == n-1 and j == n-1):
                return False
            count += 1
    return board[n-1][n-1] == 0

def get_neighbors(node):
    """Tạo các node con từ node hiện tại"""
    board = node.board
    n = len(board)
    x, y = find_blank(board)
    directions = [(-1, 0, "UP"), (1, 0, "DOWN"), (0, -1, "LEFT"), (0, 1, "RIGHT")]
    neighbors = []

    for dx, dy, move in directions:
        nx, ny = x + dx, y + dy
        if 0 <= nx < n and 0 <= ny < n:
            new_board = deepcopy(board)
            # Hoán đổi ô trống với ô lân cận
            new_board[x][y], new_board[nx][ny] = new_board[nx][ny], new_board[x][y]
            new_node = Node(new_board, node, move, node.depth + 1, node.depth + 1)
            neighbors.append(new_node)
    return neighbors

def a_star_15puzzle(initial_board):
    """Thuật toán A* giải 15-Puzzle"""
    start_node = Node(initial_board)
    start_node.heuristic = manhattan_distance(initial_board)
    start_node.f = start_node.cost + start_node.heuristic

    open_set = []                   # Priority Queue
    heapq.heappush(open_set, start_node)

    closed_set = set()              # Lưu trạng thái đã xét (tuple của tuple)

    nodes_expanded = 0

    while open_set:
        current = heapq.heappop(open_set)
        nodes_expanded += 1

        # Chuyển board thành tuple để hash
        board_tuple = tuple(tuple(row) for row in current.board)
        if board_tuple in closed_set:
            continue
        closed_set.add(board_tuple)

        if is_goal(current.board):
            # Truy vết đường đi
            path = []
            while current.parent:
                path.append(current.move)
                current = current.parent
            path.reverse()
            return path, nodes_expanded, len(closed_set)

        for neighbor in get_neighbors(current):
            neighbor.heuristic = manhattan_distance(neighbor.board)
            neighbor.f = neighbor.cost + neighbor.heuristic

            neighbor_tuple = tuple(tuple(row) for row in neighbor.board)
            if neighbor_tuple not in closed_set:
                heapq.heappush(open_set, neighbor)

    return None, nodes_expanded, len(closed_set)  # Không tìm thấy lời giải

# ====================== TEST BÀI 1 ======================
if __name__ == "__main__":
    # Trạng thái ban đầu (có thể thay đổi)
    initial = [
        [1,  2,  3,  4],
        [5,  6,  7,  8],
        [9, 10, 11, 12],
        [13, 0, 14, 15]
    ]

    print("Đang giải 15-Puzzle bằng A*...")
    solution, expanded, visited = a_star_15puzzle(initial)

    if solution:
        print("Tìm thấy lời giải!")
        print("Số nước đi:", len(solution))
        print("Các nước đi:", " -> ".join(solution))
        print("Số node mở rộng:", expanded)
    else:
        print("Không tìm thấy lời giải!")

Đang giải 15-Puzzle bằng A*...
Tìm thấy lời giải!
Số nước đi: 2
Các nước đi: RIGHT -> RIGHT
Số node mở rộng: 3


In [2]:
# ================================================
# BÀI 2: BÀI TOÁN NGƯỜI GIAO HÀNG (TSP) BẰNG THUẬT TOÁN A*
# ================================================

import heapq
from itertools import combinations
import math

class TSPNode:
    def __init__(self, current_city, visited, path_cost, path):
        self.current_city = current_city
        self.visited = visited          # Bitmask đại diện tập thành phố đã thăm
        self.path_cost = path_cost      # g(n)
        self.path = path                # Danh sách đường đi
        self.heuristic = 0              # h(n)
        self.f = 0                      # f(n) = g + h

    def __lt__(self, other):
        return self.f < other.f

def calculate_mst_lower_bound(dist, visited, n, current):
    """Heuristic: Lower bound bằng Minimum Spanning Tree của các thành phố chưa thăm"""
    unvisited = [i for i in range(n) if not (visited & (1 << i))]
    if not unvisited:
        return 0

    # Tính MST đơn giản (có thể dùng Prim hoặc Kruskal cho chính xác hơn)
    # Ở đây dùng lower bound đơn giản: khoảng cách gần nhất + gần nhì
    min1 = min2 = float('inf')
    for i in unvisited:
        d = dist[current][i]
        if d < min1:
            min2 = min1
            min1 = d
        elif d < min2:
            min2 = d

    return min1 + min2 if len(unvisited) > 1 else min1

def a_star_tsp(dist):
    """A* cho TSP - Giả sử thành phố 0 là điểm xuất phát và kết thúc"""
    n = len(dist)
    start_visited = 1 << 0                      # Đã thăm thành phố 0
    start_node = TSPNode(0, start_visited, 0, [0])
    start_node.heuristic = calculate_mst_lower_bound(dist, start_visited, n, 0)
    start_node.f = start_node.path_cost + start_node.heuristic

    open_set = []
    heapq.heappush(open_set, start_node)

    closed = set()

    while open_set:
        current = heapq.heappop(open_set)

        state = (current.current_city, current.visited)
        if state in closed:
            continue
        closed.add(state)

        # Nếu đã thăm hết tất cả thành phố → quay về 0
        if current.visited == (1 << n) - 1:
            total_cost = current.path_cost + dist[current.current_city][0]
            final_path = current.path + [0]
            return final_path, total_cost

        # Mở rộng các thành phố chưa thăm
        for next_city in range(n):
            if not (current.visited & (1 << next_city)):
                new_visited = current.visited | (1 << next_city)
                new_cost = current.path_cost + dist[current.current_city][next_city]
                new_path = current.path + [next_city]

                new_node = TSPNode(next_city, new_visited, new_cost, new_path)
                new_node.heuristic = calculate_mst_lower_bound(dist, new_visited, n, next_city)
                new_node.f = new_node.path_cost + new_node.heuristic

                heapq.heappush(open_set, new_node)

    return None, float('inf')

# ====================== TEST BÀI 2 ======================
if __name__ == "__main__":
    # Ma trận khoảng cách ví dụ (4 thành phố)
    distance_matrix = [
        [0, 10, 15, 20],
        [10, 0, 35, 25],
        [15, 35, 0, 30],
        [20, 25, 30, 0]
    ]

    print("\nĐang giải TSP bằng A*...")
    path, cost = a_star_tsp(distance_matrix)

    if path:
        print("Đường đi tối ưu:", " -> ".join(map(str, path)))
        print("Tổng chi phí:", cost)
    else:
        print("Không tìm thấy đường đi!")


Đang giải TSP bằng A*...
Đường đi tối ưu: 0 -> 1 -> 3 -> 2 -> 0
Tổng chi phí: 80
